In [ ]:
import numpy as np
import os
from cloudvolume import CloudVolume, Skeleton
import logging
import glob
import json
import neuroglancer
import requests
import tifffile
import navis
import shutil
from joblib import Parallel, delayed, parallel_config
import uuid
from natsort import natsorted

from acanalysis.skeleton_reconstruction.util import read_navis_neurons_tar, write_navis_skels_tar, swap_dimensions
from acanalysis.skeleton_reconstruction.neuroglancer import *

**Create a shareable neuroglancer links that you can use to view tiff and zarr data**

In [ ]:
ng_link = tifs_to_ngl_link(source_path = '/ACdata/Users/connorl/stacks/',  out_path = '/ACdata/Users/connorl/stacks/precomputed/')

In [ ]:
ng_link= zarr_to_ngl_link('/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S32_230412_highres/H17_x55_S32_230412_highres.zarr/highres_Pos105/', ip = 'localhost', port='9999')

**Create computed volumes for existing SWC skeletons, and load them into a neuroglancer viewer instance along with their associated zarr image files.**

In [ ]:
indir = '/ACdata/Users/connorl/Skeletons/H17_x55_S32_230412_highres_MIP1/Translated/'
outdir = '/ACdata/Users/connorl/Skeletons/H17_x55_S32_230412_highres_MIP1/Translated/Precomputed/'
files = natsorted(glob.glob(indir + "*.gz"))

#create segmentation folder
generate_ngl_segmentation_empty(outdir)

#iterate over strips and create precomputed skeletons
posns = [x.split('/')[-1].split('.')[0] for x in files]
with parallel_config(backend="loky", inner_max_num_threads=1):
    %time results = Parallel(n_jobs=15)(delayed(create_precompute_skels)(file=file,pos=pos,outpath=outdir, oid=oid, swap_dim=True) for oid,(file,pos) in enumerate(zip(files, posns)))

#extract values for generating the segmentation properties file
skel_ids = sum([x[0] for x in results],[])
tags = sum([x[1] for x in results],[])
values = sum([x[2] for x in results],[])
skel_length = sum([x[3] for x in results],[])

generate_ngl_segproperties(outdir, skel_ids=skel_ids, tags=tags, values=values, skel_length=skel_length)

**Create neuroglancer link for multiple strips and their associated skeletons.**

In [ ]:
strip_range = [0,10]
skels_dir = '/ACdata/Users/connorl/Skeletons/H17_x55_S32_230412_highres_MIP1/Downsampled/Precomputed/'
zarr_dir = '/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S32_230412_highres/H17_x55_S32_230412_highres.zarr/'

create_ngl_link_StripsWITHSkels(zarr_dir=zarr_dir, strip_range=strip_range, skels_dir=skels_dir, skel_mip=1, port='9999')